# Day 14 · Zillacode prep 3/4: joins in depth and window functions

35 of Zillacode's 52 problems join tables, and most of the Hard ones need a window function. Today covers both, in depth:
1. Join types, and what decides the number of rows
2. Same column names, one table in two roles, self-joins
3. Joining on a range, and cross joins
4. Window functions: a group's value on every row
5. Ranking: `row_number`, `rank`, `dense_rank`, top N and latest row
6. Previous and next rows, running totals and the frame trap

In Zillacode's topic list this is *Complex Joins* and *Windows Functions*. The setup cell is the same as on Days 12 and 13.

## How to work
* Run the **Setup** cell first, then the cell that prints the tables.
* **Lessons** explain one tool at a time. Read them, run every example cell (the output is already shown under it), then change the example and run it again until you can predict the result.
* **Exercises** come right after each lesson and are in Zillacode's format: complete `etl(...)` in the cell under the task so that it **returns** a DataFrame, then run the cell. The last line, `etl(...).show()`, prints your result. It must match the expected output: same columns, same rows, same order. (Zillacode itself ignores the order of rows and columns; the expected outputs here are sorted only so you can compare them by eye.)
* The answers are collapsed at the very end of the notebook. Open one only after you have tried, then close it and write the solution again yourself.
* Today: about 1 hour 15 minutes of lessons and 2 hours of exercises (19 quick ones, then 4 mixed problems).

**Setup (once):** Java 17 or 21 and Python 3.10 or newer. In a new folder: `python3.12 -m venv .venv`, `source .venv/bin/activate`, `pip install pyspark==4.2.0 jupyterlab`, then `jupyter lab` and open this notebook.

## The data
A small online shop. The setup cell creates every table from rows you can read, and the cell after it prints them.

* **customers**: *customer_id*, *name*, *email*, *city*, *signup_date*
* **products**: *product_id*, *product_name*, *category*, *price* (dollars for one item)
* **orders**: *order_id*, *customer_id*, *product_id*, *quantity*, *order_date*, *status* (`delivered`, `shipped` or `cancelled`)
* **reviews**: *review_id*, *product_id*, *customer_id*, *review_date* (text like `15/01/2024`), *rating* (1 to 5), *comment* (messy text), *tags* (text like `fast delivery;good price`)
* **signups_raw**: sign-ups exactly as a web form sent them: *signup_id*, *full_name*, *email*, *phone*, *signup_date* (text in three formats), *amount* (text), *prefs* (JSON text)
* **customers_new**: the customers table one week later
* **promotions**: *promo_id*, *category*, *start_date*, *end_date* (both days included), *discount_pct*
* **events**: website visits: *customer_id*, *event_time* (a timestamp), *page*
* **referrals**: *referrer_id* (the customer who invited someone), *referred_id* (the customer who joined), *referral_date*

The data has the traps that Zillacode's hidden tests like to use: order 1011 is a guest checkout (no *customer_id*); customer 6 has no email and no city; customer 7 has never ordered; product 107 has never been sold; order 1016 is for product 108, which is not in *products*; review 6 has no tags; two events were loaded twice; referral 5 points to customer 9, who doesn't exist.

## Setup
Run this cell once. It starts Spark and creates the tables.

In [ ]:
# Setup: starts Spark and creates the tables. Run it once.
import logging, os, sys
from datetime import date
os.environ.setdefault("PYSPARK_PYTHON", sys.executable)   # Spark's Python workers use this notebook's Python
from pyspark.sql import SparkSession, functions as F, Window as W   # the same names as in Zillacode's editor

spark = (SparkSession.builder
         .master("local[4]")
         .appName("zillacode-prep-14")
         .config("spark.sql.ansi.enabled", "false")        # behave like Zillacode (Spark 3): bad text gives null, not an error
         .config("spark.sql.shuffle.partitions", "4")      # small data: 4 shuffle partitions are enough
         .config("spark.sql.session.timeZone", "UTC")
         .config("spark.ui.showConsoleProgress", "false")
         .config("spark.log.level", "FATAL")               # hide Spark's log messages
         # Spark 4.2.0 bug (SPARK-60294): Python UDFs on Parquet can hang without this
         .config("spark.sql.execution.arrow.pythonUDF.columnarInput.enabled", "false")
         .getOrCreate())
for name in ("DataFrameQueryContextLogger", "SQLQueryContextLogger"):
    logging.getLogger(name).setLevel(logging.CRITICAL)        # errors still show, without an extra JSON log

customers = spark.createDataFrame([
    # customer_id, name, email, city, signup_date
    (1, "Aziz Karimov",     "aziz@mail.uz",     "Tashkent", date(2024, 1, 5)),
    (2, "Bella Schmidt",    "bella@web.de",     "Berlin",   date(2024, 1, 20)),
    (3, "Carlos Ruiz",      "carlos@correo.es", "Madrid",   date(2024, 2, 2)),
    (4, "Dilnoza Rahimova", "dilnoza@mail.uz",  "Tashkent", date(2024, 2, 14)),
    (5, "Emma Wilson",      "emma@post.co.uk",  "London",   date(2024, 3, 1)),
    (6, "Farid Aliyev",     None,               None,       date(2024, 3, 10)),
    (7, "Grace Lee",        "grace@web.de",     "Berlin",   date(2024, 3, 15)),
], "customer_id INT, name STRING, email STRING, city STRING, signup_date DATE")

products = spark.createDataFrame([
    # product_id, product_name, category, price (dollars for one item)
    (101, "Laptop",       "Electronics", 1200.0),
    (102, "Headphones",   "Electronics",  150.0),
    (103, "Coffee Maker", "Kitchen",       85.0),
    (104, "Blender",      "Kitchen",       60.0),
    (105, "Novel",        "Books",         15.0),
    (106, "Cookbook",     "Books",         25.0),
    (107, "Desk Lamp",    "Home",          40.0),
], "product_id INT, product_name STRING, category STRING, price DOUBLE")

orders = spark.createDataFrame([
    # order_id, customer_id, product_id, quantity, order_date, status
    (1001, 1,    101, 1, date(2024, 1, 10), "delivered"),
    (1002, 1,    105, 3, date(2024, 1, 10), "delivered"),
    (1003, 2,    102, 2, date(2024, 1, 25), "delivered"),
    (1004, 3,    103, 1, date(2024, 2, 3),  "cancelled"),
    (1005, 2,    106, 1, date(2024, 2, 5),  "delivered"),
    (1006, 4,    104, 2, date(2024, 2, 15), "shipped"),
    (1007, 1,    102, 1, date(2024, 2, 20), "delivered"),
    (1008, 5,    101, 1, date(2024, 3, 2),  "shipped"),
    (1009, 3,    105, 4, date(2024, 3, 5),  "delivered"),
    (1010, 4,    106, 2, date(2024, 3, 8),  "cancelled"),
    (1011, None, 104, 1, date(2024, 3, 9),  "delivered"),
    (1012, 6,    103, 2, date(2024, 3, 12), "shipped"),
    (1013, 2,    101, 1, date(2024, 3, 18), "delivered"),
    (1014, 5,    105, 5, date(2024, 3, 20), "delivered"),
    (1015, 1,    103, 1, date(2024, 3, 25), "shipped"),
    (1016, 3,    108, 1, date(2024, 3, 28), "delivered"),
], "order_id INT, customer_id INT, product_id INT, quantity INT, order_date DATE, status STRING")

reviews = spark.createDataFrame([
    # review_id, product_id, customer_id, review_date (text), rating, comment, tags (text)
    (1, 101, 1, "15/01/2024", 5, "  Great laptop, very FAST!  ",    "fast delivery;good price"),
    (2, 105, 1, "20/01/2024", 4, "Nice story.",                     "good price"),
    (3, 102, 2, "02/02/2024", 3, "Sound is ok, delivery was slow",  "slow delivery"),
    (4, 103, 3, "10/02/2024", 1, "Broke after one week!!",          "broken;slow delivery"),
    (5, 101, 5, "09/03/2024", 5, "Fast and powerful",               "fast delivery"),
    (6, 105, 3, "12/03/2024", 2, "Too long, boring",                None),
    (7, 106, 2, "25/03/2024", 4, "Useful recipes",                  "good price;gift"),
    (8, 103, 6, "30/03/2024", 4, "Good coffee, slow start",         "gift"),
], "review_id INT, product_id INT, customer_id INT, review_date STRING, rating INT, comment STRING, tags STRING")

signups_raw = spark.createDataFrame([
    # signup_id, full_name, email, phone, signup_date (text, three formats), amount (text), prefs (JSON text)
    (1, "  aziz   KARIMOV ", "Aziz@Mail.UZ ",      "+998 90 123-45-67", "2024-01-05", "12.50", '{"newsletter": true, "lang": "uz"}'),
    (2, "bella schmidt",     "bella@web.de",       "0049 151 2345678",  "20/01/2024", "abc",   '{"newsletter": false, "lang": "de"}'),
    (3, "Carlos  Ruiz",      "",                   "(34) 612 345 678",  "2024/02/02", "7",     '{"lang": "es"}'),
    (4, "DILNOZA rahimova",  "dilnoza@mail.uz",    "90-765-43-21",      "14/02/2024", "",      None),
    (5, "Emma Wilson",       "emma[at]post.co.uk", "07700 900123",      "2024-03-01", "3.75",  '{"newsletter": true}'),
    (6, "aziz karimov",      "aziz@mail.uz",       "+998901234567",     "2024-03-02", "1.00",  '{"newsletter": false, "lang": "uz"}'),
], "signup_id INT, full_name STRING, email STRING, phone STRING, signup_date STRING, amount STRING, prefs STRING")

customers_new = spark.createDataFrame([
    # the customers table one week later: 2 moved, 6 has an email now, 7 is gone, 8 is new
    (1, "Aziz Karimov",     "aziz@mail.uz",     "Tashkent", date(2024, 1, 5)),
    (2, "Bella Schmidt",    "bella@web.de",     "Munich",   date(2024, 1, 20)),
    (3, "Carlos Ruiz",      "carlos@correo.es", "Madrid",   date(2024, 2, 2)),
    (4, "Dilnoza Rahimova", "dilnoza@mail.uz",  "Tashkent", date(2024, 2, 14)),
    (5, "Emma Wilson",      "emma@post.co.uk",  "London",   date(2024, 3, 1)),
    (6, "Farid Aliyev",     "farid@mail.az",    None,       date(2024, 3, 10)),
    (8, "Hana Sato",        "hana@mail.jp",     "Osaka",    date(2024, 4, 2)),
], customers.schema)

promotions = spark.createDataFrame([
    # promo_id, category, start_date, end_date (both days included), discount_pct
    ("WINTER",   "Electronics", date(2024, 1, 1),  date(2024, 1, 31), 10),
    ("BOOKWEEK", "Books",       date(2024, 3, 1),  date(2024, 3, 10), 20),
    ("SPRING",   "Kitchen",     date(2024, 3, 1),  date(2024, 3, 31), 15),
    ("FLASH",    "Electronics", date(2024, 3, 15), date(2024, 3, 20), 5),
], "promo_id STRING, category STRING, start_date DATE, end_date DATE, discount_pct INT")

events = spark.createDataFrame([
    # customer_id, event_time, page (two rows were loaded twice)
    (1, "2024-03-01 09:00:00", "home"),
    (1, "2024-03-01 09:10:00", "product"),
    (1, "2024-03-01 09:10:00", "product"),
    (1, "2024-03-01 11:00:00", "home"),
    (1, "2024-03-02 10:00:00", "cart"),
    (1, "2024-03-03 10:30:00", "home"),
    (1, "2024-03-05 08:00:00", "home"),
    (2, "2024-03-01 20:00:00", "home"),
    (2, "2024-03-01 20:20:00", "product"),
    (2, "2024-03-01 20:45:00", "cart"),
    (2, "2024-03-04 18:00:00", "home"),
    (2, "2024-03-05 18:00:00", "home"),
    (2, "2024-03-05 18:00:00", "home"),
    (3, "2024-03-02 12:00:00", "product"),
    (3, "2024-03-03 12:00:00", "product"),
    (3, "2024-03-04 12:00:00", "cart"),
], "customer_id INT, event_time STRING, page STRING").withColumn("event_time", F.to_timestamp("event_time"))

referrals = spark.createDataFrame([
    # referrer_id (the customer who invited), referred_id (the customer who joined), referral_date
    (1, 4, date(2024, 2, 10)),
    (4, 5, date(2024, 2, 25)),
    (1, 6, date(2024, 3, 8)),
    (2, 7, date(2024, 3, 12)),
    (2, 9, date(2024, 3, 30)),
], "referrer_id INT, referred_id INT, referral_date DATE")

print("Spark", spark.version, "is ready. Tables: customers, products, orders, reviews, signups_raw, customers_new, "
      "promotions, events, referrals")

Spark 4.2.0 is ready. Tables: customers, products, orders, reviews, signups_raw, customers_new, promotions, events, referrals


Print every table, so you can check results by eye:

In [ ]:
for name in ["customers", "products", "orders", "reviews", "signups_raw", "customers_new", "promotions", "events", "referrals"]:
    print(name)
    globals()[name].show(truncate=False)

customers
+-----------+----------------+----------------+--------+-----------+
|customer_id|name            |email           |city    |signup_date|
+-----------+----------------+----------------+--------+-----------+
|1          |Aziz Karimov    |aziz@mail.uz    |Tashkent|2024-01-05 |
|2          |Bella Schmidt   |bella@web.de    |Berlin  |2024-01-20 |
|3          |Carlos Ruiz     |carlos@correo.es|Madrid  |2024-02-02 |
|4          |Dilnoza Rahimova|dilnoza@mail.uz |Tashkent|2024-02-14 |
|5          |Emma Wilson     |emma@post.co.uk |London  |2024-03-01 |
|6          |Farid Aliyev    |NULL            |NULL    |2024-03-10 |
|7          |Grace Lee       |grace@web.de    |Berlin  |2024-03-15 |
+-----------+----------------+----------------+--------+-----------+

products
+----------+------------+-----------+------+
|product_id|product_name|category   |price |
+----------+------------+-----------+------+
|101       |Laptop      |Electronics|1200.0|
|102       |Headphones  |Electronics|150.

## Lesson 1 · Join types

`left.join(right, on, how)`:

| `how` | Keeps |
|---|---|
| `"inner"` (the default) | the rows that have a match on both sides |
| `"left"` (also `"left_outer"`) | every left row; the right columns are null where nothing matched |
| `"right"` | every right row |
| `"full"` (also `"outer"`, `"full_outer"`) | every row of both sides |
| `"left_semi"` | the left rows that **have** a match: only the left columns, and never duplicated (SQL `WHERE EXISTS`) |
| `"left_anti"` | the left rows that have **no** match: only the left columns (SQL `WHERE NOT EXISTS`) |
| `a.crossJoin(b)` | every combination: rows of a × rows of b (Lesson 3) |

**Three ways to write `on`:**

| `on=` | Result |
|---|---|
| `"customer_id"` | the same name on both sides; the output has **one** customer_id column |
| `["customer_id", "day"]` | several keys with the same names |
| `orders.customer_id == customers.customer_id`, or with different names `mines.id == extraction.mine_id` | an expression; the output keeps **both** key columns (Lesson 2) |

**What decides the number of rows:**
* **A null key never matches**, not even another null. The guest order (no customer_id) drops out of every inner join on customer_id.
* **One-to-many repeats rows**: a customer with 4 orders appears 4 times after the join (and Day 13, Lesson 3: two "many" tables multiply).
* **After a full join** on an expression, the key of the side without a match is null: combine with `F.coalesce(a.key, b.key)`. With `on="key"`, Spark does that for you.

On Zillacode, an "all X, even without Y" sentence means a left join; "only X that have Y" means inner or left_semi; "X that never..." means left_anti; "all records from both" means full (#14, #24, #47). [join docs](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.join.html).

In [ ]:
c = customers.select("customer_id", "name")
o = orders.select("order_id", "customer_id", "status")
for how in ["inner", "left", "right", "full", "left_semi", "left_anti"]:
    print(f"{how:10}", c.join(o, "customer_id", how).count(), "rows")

inner      15 rows
left       16 rows
right      16 rows
full       17 rows
left_semi  6 rows
left_anti  1 rows


In [ ]:
# left: Grace (7) has no orders, so her order columns are null. right: the guest order has no customer.
c.join(o, "customer_id", "left").filter(F.col("customer_id") >= 6).orderBy("customer_id").show()
c.join(o, "customer_id", "right").filter(F.col("customer_id").isNull()).show()

+-----------+------------+--------+-------+
|customer_id|        name|order_id| status|
+-----------+------------+--------+-------+
|          6|Farid Aliyev|    1012|shipped|
|          7|   Grace Lee|    NULL|   NULL|
+-----------+------------+--------+-------+

+-----------+----+--------+---------+
|customer_id|name|order_id|   status|
+-----------+----+--------+---------+
|       NULL|NULL|    1011|delivered|
+-----------+----+--------+---------+


In [ ]:
# semi and anti: only the left columns, at most one row per left row
c.join(o, "customer_id", "left_semi").orderBy("customer_id").show()
c.join(o, "customer_id", "left_anti").show()

+-----------+----------------+
|customer_id|            name|
+-----------+----------------+
|          1|    Aziz Karimov|
|          2|   Bella Schmidt|
|          3|     Carlos Ruiz|
|          4|Dilnoza Rahimova|
|          5|     Emma Wilson|
|          6|    Farid Aliyev|
+-----------+----------------+

+-----------+---------+
|customer_id|     name|
+-----------+---------+
|          7|Grace Lee|
+-----------+---------+


### Exercises

**Q1 · `etl(customers, orders)`** Which customers have never placed an order? Show customer_id and name.

Expected output:

```
+-----------+---------+
|customer_id|     name|
+-----------+---------+
|          7|Grace Lee|
+-----------+---------+
```

In [ ]:
def etl(customers, orders):
    # Write code here: return a DataFrame
    return customers

etl(customers, orders).show()

**Q2 · `etl(customers, orders)`** Show every customer with their number of orders, including customers with 0 orders. Show name and orders, most orders first; on a tie, by name.

Expected output:

```
+----------------+------+
|            name|orders|
+----------------+------+
|    Aziz Karimov|     4|
|   Bella Schmidt|     3|
|     Carlos Ruiz|     3|
|Dilnoza Rahimova|     2|
|     Emma Wilson|     2|
|    Farid Aliyev|     1|
|       Grace Lee|     0|
+----------------+------+
```

In [ ]:
def etl(customers, orders):
    # Write code here: return a DataFrame
    return customers

etl(customers, orders).show()

**Q3 · `etl(orders, products)`** Show every product of the catalog and every product_id that appears in orders, with its number of orders (0 when none). Products missing from the catalog show an empty name. Columns: product_id, product_name, orders; sorted by product_id.

Expected output:

```
+----------+------------+------+
|product_id|product_name|orders|
+----------+------------+------+
|       101|      Laptop|     3|
|       102|  Headphones|     2|
|       103|Coffee Maker|     3|
|       104|     Blender|     2|
|       105|       Novel|     3|
|       106|    Cookbook|     2|
|       107|   Desk Lamp|     0|
|       108|        NULL|     1|
+----------+------------+------+
```

In [ ]:
def etl(orders, products):
    # Write code here: return a DataFrame
    return orders

etl(orders, products).show()

**Q4 · `etl(orders, customers)`** Which customers had all of their orders delivered (and at least one order)? Show their names, from A to Z.

Expected output:

```
+-------------+
|         name|
+-------------+
|Bella Schmidt|
+-------------+
```

In [ ]:
def etl(orders, customers):
    # Write code here: return a DataFrame
    return orders

etl(orders, customers).show()

## Lesson 2 · Same column names, one table in two roles, self-joins

After a join on an expression, the columns of **both** sides are there. If both sides have a column with the same name (`id`, `name`, `customer_id`), the plain name is **ambiguous**, and Spark stops with `[AMBIGUOUS_REFERENCE]`. Three fixes:

1. **Pick through the DataFrame**: `select(stars["name"].alias("star_name"), planets["name"].alias("planet_name"))`. Zillacode's #33 and #52 do this.
2. **Alias the DataFrames** and use dotted names: `a = df.alias("a")`, then `F.col("a.name")`. #27 and #42 do this. It's the only way in a **self-join**, where both sides are the same DataFrame.
3. **Rename before the join**, so there is nothing to confuse: `airports.withColumnRenamed("airport_name", "origin_name")`. #49 does this. It's the cleanest choice when one table plays **two roles**.

**One table in two roles** (origin and destination airport; the customer who invited and the customer who joined) means two joins with two renamed copies of the same table.

**A self-join** compares rows of one table with each other: pairs in the same city, employees and their managers, an order and the same customer's earlier orders. To get each pair once, and never pair a row with itself, add `a.id < b.id` to the condition.

In [ ]:
# Ambiguous: both sides have customer_id after a join on an expression
try:
    c.join(o, c.customer_id == o.customer_id).select("customer_id").show()
except Exception as e:
    print(type(e).__name__, "->", str(e).split(".")[0])
# Fix 1: say which side you mean
c.join(o, c.customer_id == o.customer_id).select(c["customer_id"], "name", "order_id").orderBy("order_id").show(3)

AnalysisException -> [AMBIGUOUS_REFERENCE] Reference `customer_id` is ambiguous, could be: [`customer_id`, `customer_id`]
+-----------+-------------+--------+
|customer_id|         name|order_id|
+-----------+-------------+--------+
|          1| Aziz Karimov|    1001|
|          1| Aziz Karimov|    1002|
|          2|Bella Schmidt|    1003|
+-----------+-------------+--------+
only showing top 3 rows


In [ ]:
# One table in two roles: rename it once per role, then join twice
airports = spark.createDataFrame([("TAS", "Tashkent"), ("IST", "Istanbul"), ("BER", "Berlin")], "code string, city string")
flights = spark.createDataFrame([(1, "TAS", "IST"), (2, "IST", "BER"), (3, "BER", "TAS")], "flight_id int, origin string, destination string")
origin = airports.select(F.col("code").alias("origin"), F.col("city").alias("origin_city"))
destination = airports.select(F.col("code").alias("destination"), F.col("city").alias("destination_city"))
(flights.join(origin, "origin")
        .join(destination, "destination")
        .select("flight_id", "origin_city", "destination_city")
        .orderBy("flight_id")
        .show())

+---------+-----------+----------------+
|flight_id|origin_city|destination_city|
+---------+-----------+----------------+
|        1|   Tashkent|        Istanbul|
|        2|   Istanbul|          Berlin|
|        3|     Berlin|        Tashkent|
+---------+-----------+----------------+


In [ ]:
# Self-join: pairs of products in the same category, each pair once
a, b = products.alias("a"), products.alias("b")
(a.join(b, (F.col("a.category") == F.col("b.category")) & (F.col("a.product_id") < F.col("b.product_id")))
  .select(F.col("a.category").alias("category"),
          F.col("a.product_name").alias("product_a"),
          F.col("b.product_name").alias("product_b"))
  .orderBy("category")
  .show())

+-----------+------------+----------+
|   category|   product_a| product_b|
+-----------+------------+----------+
|      Books|       Novel|  Cookbook|
|Electronics|      Laptop|Headphones|
|    Kitchen|Coffee Maker|   Blender|
+-----------+------------+----------+


### Exercises

**Q5 · `etl(referrals, customers)`** Show every referral with both names: referral_date, referrer (the name of the customer who invited) and referred (the name of the customer who joined; empty if unknown). Sort by referral_date.

Expected output:

```
+-------------+----------------+----------------+
|referral_date|        referrer|        referred|
+-------------+----------------+----------------+
|   2024-02-10|    Aziz Karimov|Dilnoza Rahimova|
|   2024-02-25|Dilnoza Rahimova|     Emma Wilson|
|   2024-03-08|    Aziz Karimov|    Farid Aliyev|
|   2024-03-12|   Bella Schmidt|       Grace Lee|
|   2024-03-30|   Bella Schmidt|            NULL|
+-------------+----------------+----------------+
```

In [ ]:
def etl(referrals, customers):
    # Write code here: return a DataFrame
    return referrals

etl(referrals, customers).show()

**Q6 · `etl(customers)`** Show the pairs of customers who live in the same city: city, customer_a and customer_b (names). Each pair once, customer_a being the one with the smaller customer_id. Sort by city.

Expected output:

```
+--------+-------------+----------------+
|    city|   customer_a|      customer_b|
+--------+-------------+----------------+
|  Berlin|Bella Schmidt|       Grace Lee|
|Tashkent| Aziz Karimov|Dilnoza Rahimova|
+--------+-------------+----------------+
```

In [ ]:
def etl(customers):
    # Write code here: return a DataFrame
    return customers

etl(customers).show()

**Q7 · `etl(customers, customers_new)`** Compare `customers` with `customers_new`: which customers were `added`, `removed` or `changed` (a different name, email or city)? Show customer_id and change, sorted by customer_id.

Expected output:

```
+-----------+-------+
|customer_id| change|
+-----------+-------+
|          2|changed|
|          6|changed|
|          7|removed|
|          8|  added|
+-----------+-------+
```

In [ ]:
def etl(customers, customers_new):
    # Write code here: return a DataFrame
    return customers

etl(customers, customers_new).show()

## Lesson 3 · Joining on a range, and cross joins

A join condition can be any true/false expression, not only `==`:
* **A date inside a period**: `F.col("order_date").between(F.col("start_date"), F.col("end_date"))`, usually together with an equality (`products.category == promotions.category`). Zillacode's #22 (minutes watched while a subscription was active) is this pattern.
* **A value inside a band**: price bands, tax brackets, age groups.

A join on equality plus a range is fast. A join on a range alone compares every pair of rows, which is fine for small tables like Zillacode's.

`crossJoin` pairs every row of one side with every row of the other (Zillacode #38). Its typical job is to **build a complete grid** (every customer × every category, every day × every store), then `left` join the real numbers onto it and fill the gaps with 0. That way, combinations without any data still appear in the answer.

In [ ]:
bands = spark.createDataFrame([("budget", 0.0, 49.99), ("standard", 50.0, 199.99), ("premium", 200.0, 99999.0)],
                              "band string, low double, high double")
(products.join(bands, F.col("price").between(F.col("low"), F.col("high")))
         .select("product_name", "price", "band")
         .orderBy("price")
         .show())

+------------+------+--------+
|product_name| price|    band|
+------------+------+--------+
|       Novel|  15.0|  budget|
|    Cookbook|  25.0|  budget|
|   Desk Lamp|  40.0|  budget|
|     Blender|  60.0|standard|
|Coffee Maker|  85.0|standard|
|  Headphones| 150.0|standard|
|      Laptop|1200.0| premium|
+------------+------+--------+


In [ ]:
# A full grid of month × status, then the real counts, then 0 for the gaps
months = orders.select(F.month("order_date").alias("month")).distinct()
statuses = orders.select("status").distinct()
counts = orders.groupBy(F.month("order_date").alias("month"), "status").agg(F.count("*").alias("orders"))
(months.crossJoin(statuses)
       .join(counts, ["month", "status"], "left")
       .fillna(0)
       .orderBy("month", "status")
       .show())

+-----+---------+------+
|month|   status|orders|
+-----+---------+------+
|    1|cancelled|     0|
|    1|delivered|     3|
|    1|  shipped|     0|
|    2|cancelled|     1|
|    2|delivered|     2|
|    2|  shipped|     1|
|    3|cancelled|     1|
|    3|delivered|     5|
|    3|  shipped|     3|
+-----+---------+------+


### Exercises

**Q8 · `etl(orders, products, promotions)`** For each delivered order, find the promotion of its product's category that was running on the order date. Show order_id, product_name, promo_id and discount_pct; orders without a promotion drop out. Sort by order_id.

Expected output:

```
+--------+------------+--------+------------+
|order_id|product_name|promo_id|discount_pct|
+--------+------------+--------+------------+
|    1001|      Laptop|  WINTER|          10|
|    1003|  Headphones|  WINTER|          10|
|    1009|       Novel|BOOKWEEK|          20|
|    1011|     Blender|  SPRING|          15|
|    1013|      Laptop|   FLASH|           5|
+--------+------------+--------+------------+
```

In [ ]:
def etl(orders, products, promotions):
    # Write code here: return a DataFrame
    return orders

etl(orders, products, promotions).show()

**Q9 · `etl(customers, products, orders)`** Make a table with every combination of customers 1 to 3 and every product category, with the number of orders (0 when none). Columns: customer_id, category, orders; sorted by customer_id, then category.

Expected output:

```
+-----------+-----------+------+
|customer_id|   category|orders|
+-----------+-----------+------+
|          1|      Books|     1|
|          1|Electronics|     2|
|          1|       Home|     0|
|          1|    Kitchen|     1|
|          2|      Books|     1|
|          2|Electronics|     2|
|          2|       Home|     0|
|          2|    Kitchen|     0|
|          3|      Books|     1|
|          3|Electronics|     0|
|          3|       Home|     0|
|          3|    Kitchen|     1|
+-----------+-----------+------+
```

In [ ]:
def etl(customers, products, orders):
    # Write code here: return a DataFrame
    return customers

etl(customers, products, orders).show()

## Lesson 4 · Window functions: a group's value on every row

`groupBy` turns each group into **one** row. A **window function** also computes over a group of rows, but it **keeps every row** and writes the result next to it:

```python
w = W.partitionBy("category")                       # the group: rows with the same category
products.withColumn("category_avg", F.avg("price").over(w))
```

* `W.partitionBy(cols)` decides which rows belong together, like `groupBy`. With **no** `partitionBy`, the whole table is one window. That's fine for small data; Spark warns that it moves every row into one partition.
* `.orderBy(cols)` sets the order inside each window. Ranks, `lag`/`lead` and running totals need it (Lessons 5 and 6).
* **Any aggregate** works with `.over(w)`: `sum`, `avg`, `min`, `max`, `count`, `variance` (Zillacode #29)...

Typical uses without an order: compare each row with its group (price minus the category average), a share of the group (`F.col("x") / F.sum("x").over(w)`), the group's size next to each row, and keeping the rows that equal the group's maximum (ties all stay). [Window docs](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/window.html).

In [ ]:
w = W.partitionBy("category")
products.select(
    "product_name", "category", "price",
    F.round(F.avg("price").over(w), 2).alias("category_avg"),
    F.max("price").over(w).alias("category_max"),
    F.count("*").over(w).alias("in_category"),
    F.round(F.col("price") / F.sum("price").over(w) * 100, 1).alias("pct_of_category"),
).orderBy("category", "price").show()

+------------+-----------+------+------------+------------+-----------+---------------+
|product_name|   category| price|category_avg|category_max|in_category|pct_of_category|
+------------+-----------+------+------------+------------+-----------+---------------+
|       Novel|      Books|  15.0|        20.0|        25.0|          2|           37.5|
|    Cookbook|      Books|  25.0|        20.0|        25.0|          2|           62.5|
|  Headphones|Electronics| 150.0|       675.0|      1200.0|          2|           11.1|
|      Laptop|Electronics|1200.0|       675.0|      1200.0|          2|           88.9|
|   Desk Lamp|       Home|  40.0|        40.0|        40.0|          1|          100.0|
|     Blender|    Kitchen|  60.0|        72.5|        85.0|          2|           41.4|
|Coffee Maker|    Kitchen|  85.0|        72.5|        85.0|          2|           58.6|
+------------+-----------+------+------------+------------+-----------+---------------+


In [ ]:
# Keep the rows that equal their group's maximum (ties all stay)
(orders.filter(F.col("customer_id").isNotNull())
       .withColumn("max_qty", F.max("quantity").over(W.partitionBy("customer_id")))
       .filter(F.col("quantity") == F.col("max_qty"))
       .select("customer_id", "order_id", "quantity")
       .orderBy("customer_id", "order_id")
       .show())

+-----------+--------+--------+
|customer_id|order_id|quantity|
+-----------+--------+--------+
|          1|    1002|       3|
|          2|    1003|       2|
|          3|    1009|       4|
|          4|    1006|       2|
|          4|    1010|       2|
|          5|    1014|       5|
|          6|    1012|       2|
+-----------+--------+--------+


### Exercises

**Q10 · `etl(products)`** Show product_name, category, price and diff_from_avg (the price minus its category's average price, 2 decimals), sorted by category, then price.

Expected output:

```
+------------+-----------+------+-------------+
|product_name|   category| price|diff_from_avg|
+------------+-----------+------+-------------+
|       Novel|      Books|  15.0|         -5.0|
|    Cookbook|      Books|  25.0|          5.0|
|  Headphones|Electronics| 150.0|       -525.0|
|      Laptop|Electronics|1200.0|        525.0|
|   Desk Lamp|       Home|  40.0|          0.0|
|     Blender|    Kitchen|  60.0|        -12.5|
|Coffee Maker|    Kitchen|  85.0|         12.5|
+------------+-----------+------+-------------+
```

In [ ]:
def etl(products):
    # Write code here: return a DataFrame
    return products

etl(products).show()

**Q11 · `etl(orders)`** For the orders of customers 1 and 2, show order_id, customer_id, quantity and pct_of_customer: the order's share of all items that customer ordered, in percent, 1 decimal. Sort by order_id.

Expected output:

```
+--------+-----------+--------+---------------+
|order_id|customer_id|quantity|pct_of_customer|
+--------+-----------+--------+---------------+
|    1001|          1|       1|           16.7|
|    1002|          1|       3|           50.0|
|    1003|          2|       2|           50.0|
|    1005|          2|       1|           25.0|
|    1007|          1|       1|           16.7|
|    1013|          2|       1|           25.0|
|    1015|          1|       1|           16.7|
+--------+-----------+--------+---------------+
```

In [ ]:
def etl(orders):
    # Write code here: return a DataFrame
    return orders

etl(orders).show()

## Lesson 5 · Ranking: `row_number`, `rank`, `dense_rank`

With an `orderBy` in the window, you can number the rows of each partition:

| quantity | `row_number` | `rank` | `dense_rank` |
|---|---|---|---|
| 5 | 1 | 1 | 1 |
| 3 | 2 | 2 | 2 |
| 3 | 3 | 2 | 2 |
| 1 | 4 | 4 | 3 |

* `F.row_number()`: 1, 2, 3, ..., always unique. Ties get an **arbitrary** order unless you add a tie-breaker column to `orderBy`.
* `F.rank()`: ties share a number, and the next numbers are skipped (Olympic ranking).
* `F.dense_rank()`: ties share a number, nothing is skipped. "Without gaps in the ranking" in a Zillacode text means `dense_rank` (#7).
* `F.ntile(4)` splits each partition into 4 buckets; `F.percent_rank()` goes from 0 to 1.

**Recipes**
* **Top N per group**: rank over `W.partitionBy(group).orderBy(F.desc(value))`, then keep `<= N`. If ties should all stay, use `rank` or `dense_rank`; for exactly N rows, use `row_number` with a tie-breaker.
* **The latest (or first) row per key**, and **removing duplicates while choosing which row stays**: `row_number` over `partitionBy(key).orderBy(F.desc(time))`, then keep `== 1` (Zillacode #13, #30, #37).
* **Numbering the whole table**: `W.orderBy(...)` with no partition (#28, #43).

Always **drop the helper column** (`rn`, `rank`) before returning, unless the output schema asks for it: an extra column fails the checker.

In [ ]:
# Ties: orders 1006 and 1010 both have quantity 2, and five orders have quantity 1
w = W.orderBy(F.desc("quantity"))
w_unique = W.orderBy(F.desc("quantity"), "order_id")     # a tie-breaker makes row_number repeatable
(orders.filter(F.col("customer_id").isin(1, 3, 4))
       .select("order_id", "quantity",
               F.row_number().over(w_unique).alias("row_number"),
               F.rank().over(w).alias("rank"),
               F.dense_rank().over(w).alias("dense_rank"))
       .orderBy("row_number")
       .show())

+--------+--------+----------+----+----------+
|order_id|quantity|row_number|rank|dense_rank|
+--------+--------+----------+----+----------+
|    1009|       4|         1|   1|         1|
|    1002|       3|         2|   2|         2|
|    1006|       2|         3|   3|         3|
|    1010|       2|         4|   3|         3|
|    1001|       1|         5|   5|         4|
|    1004|       1|         6|   5|         4|
|    1007|       1|         7|   5|         4|
|    1015|       1|         8|   5|         4|
|    1016|       1|         9|   5|         4|
+--------+--------+----------+----+----------+


In [ ]:
# The FIRST order of each customer: row_number per customer, oldest first, keep 1, drop the helper
w = W.partitionBy("customer_id").orderBy("order_date", "order_id")
(orders.filter(F.col("customer_id").isNotNull())
       .withColumn("rn", F.row_number().over(w))
       .filter(F.col("rn") == 1)
       .drop("rn")
       .orderBy("customer_id")
       .show())

+--------+-----------+----------+--------+----------+---------+
|order_id|customer_id|product_id|quantity|order_date|   status|
+--------+-----------+----------+--------+----------+---------+
|    1001|          1|       101|       1|2024-01-10|delivered|
|    1003|          2|       102|       2|2024-01-25|delivered|
|    1004|          3|       103|       1|2024-02-03|cancelled|
|    1006|          4|       104|       2|2024-02-15|  shipped|
|    1008|          5|       101|       1|2024-03-02|  shipped|
|    1012|          6|       103|       2|2024-03-12|  shipped|
+--------+-----------+----------+--------+----------+---------+


### Exercises

**Q12 · `etl(products)`** Show the most expensive product of each category: category, product_name and price, sorted by category.

Expected output:

```
+-----------+------------+------+
|   category|product_name| price|
+-----------+------------+------+
|      Books|    Cookbook|  25.0|
|Electronics|      Laptop|1200.0|
|       Home|   Desk Lamp|  40.0|
|    Kitchen|Coffee Maker|  85.0|
+-----------+------------+------+
```

In [ ]:
def etl(products):
    # Write code here: return a DataFrame
    return products

etl(products).show()

**Q13 · `etl(orders, customers)`** Rank the customers who placed orders by their number of orders. Customers with the same number share a rank, and no rank is skipped after a tie. Show name, orders and rank, sorted by rank, then name.

Expected output:

```
+----------------+------+----+
|            name|orders|rank|
+----------------+------+----+
|    Aziz Karimov|     4|   1|
|   Bella Schmidt|     3|   2|
|     Carlos Ruiz|     3|   2|
|Dilnoza Rahimova|     2|   3|
|     Emma Wilson|     2|   3|
|    Farid Aliyev|     1|   4|
+----------------+------+----+
```

In [ ]:
def etl(orders, customers):
    # Write code here: return a DataFrame
    return orders

etl(orders, customers).show()

**Q14 · `etl(orders, products)`** Show the top 2 products of each category by items delivered (on a tie, the smaller product_id first). Columns: category, product_name, items and rank; sorted by category, then rank.

Expected output:

```
+-----------+------------+-----+----+
|   category|product_name|items|rank|
+-----------+------------+-----+----+
|      Books|       Novel|   12|   1|
|      Books|    Cookbook|    1|   2|
|Electronics|  Headphones|    3|   1|
|Electronics|      Laptop|    2|   2|
|    Kitchen|     Blender|    1|   1|
+-----------+------------+-----+----+
```

In [ ]:
def etl(orders, products):
    # Write code here: return a DataFrame
    return orders

etl(orders, products).show()

**Q15 · `etl(orders)`** Show each customer's latest order: customer_id, order_id, order_date and status (on the same day, the larger order_id). Leave out the guest order. Sort by customer_id.

Expected output:

```
+-----------+--------+----------+---------+
|customer_id|order_id|order_date|   status|
+-----------+--------+----------+---------+
|          1|    1015|2024-03-25|  shipped|
|          2|    1013|2024-03-18|delivered|
|          3|    1016|2024-03-28|delivered|
|          4|    1010|2024-03-08|cancelled|
|          5|    1014|2024-03-20|delivered|
|          6|    1012|2024-03-12|  shipped|
+-----------+--------+----------+---------+
```

In [ ]:
def etl(orders):
    # Write code here: return a DataFrame
    return orders

etl(orders).show()

## Lesson 6 · Previous and next rows, running totals

| Expression | Gives |
|---|---|
| `F.lag("x").over(w)` | x of the **previous** row in the window, null for the first row. `F.lag("x", 2)` goes two rows back; `F.lag("x", 1, 0)` gives 0 instead of null. |
| `F.lead("x").over(w)` | x of the **next** row |
| `F.sum("x").over(w.rowsBetween(W.unboundedPreceding, W.currentRow))` | a running total |
| `F.avg("x").over(w.rowsBetween(-2, W.currentRow))` | a moving average of this row and the 2 before it |
| `F.first("x").over(w)`, `F.last("x", ignorenulls=True).over(w)` | the first / last value in the frame |

Typical uses: the change since the previous row (`x - lag(x)`), growth in percent (`(x - lag(x)) / lag(x) * 100`, Zillacode #45), days since the previous order, the previous product of the same customer (#21), detecting a change (`status != lag(status)`), running balances.

**The frame trap.** Once a window has an `orderBy`, an aggregate over it uses a default **frame**: from the first row up to the current row, **plus every row with the same order value** (`rangeBetween(W.unboundedPreceding, W.currentRow)`). So two orders on the same date both get a running total that already includes each other. For a true row-by-row running total, use `rowsBetween(W.unboundedPreceding, W.currentRow)` and a tie-breaker in `orderBy`. For the same reason, `F.last("x").over(w)` with an `orderBy` returns the **current** row's value, because the frame ends there; use `rowsBetween(W.unboundedPreceding, W.unboundedFollowing)` to look at the whole partition. Without an `orderBy`, the frame is always the whole partition (Lesson 4).

In [ ]:
w = W.partitionBy("customer_id").orderBy("order_date", "order_id")
orders.filter(F.col("customer_id") == 1).select(
    "order_id", "order_date", "quantity",
    F.lag("quantity").over(w).alias("prev_qty"),
    (F.col("quantity") - F.lag("quantity").over(w)).alias("change"),
    F.lead("order_date").over(w).alias("next_date"),
    F.datediff("order_date", F.lag("order_date").over(w)).alias("days_since_prev"),
).orderBy("order_date", "order_id").show()

+--------+----------+--------+--------+------+----------+---------------+
|order_id|order_date|quantity|prev_qty|change| next_date|days_since_prev|
+--------+----------+--------+--------+------+----------+---------------+
|    1001|2024-01-10|       1|    NULL|  NULL|2024-01-10|           NULL|
|    1002|2024-01-10|       3|       1|     2|2024-02-20|              0|
|    1007|2024-02-20|       1|       3|    -2|2024-03-25|             41|
|    1015|2024-03-25|       1|       1|     0|      NULL|             34|
+--------+----------+--------+--------+------+----------+---------------+


In [ ]:
# Orders 1001 and 1002 share a date: the default frame counts them together
w_date = W.partitionBy("customer_id").orderBy("order_date")
w_rows = W.partitionBy("customer_id").orderBy("order_date", "order_id").rowsBetween(W.unboundedPreceding, W.currentRow)
orders.filter(F.col("customer_id") == 1).select(
    "order_id", "order_date", "quantity",
    F.sum("quantity").over(w_date).alias("default_frame"),
    F.sum("quantity").over(w_rows).alias("rows_frame"),
).orderBy("order_date", "order_id").show()

+--------+----------+--------+-------------+----------+
|order_id|order_date|quantity|default_frame|rows_frame|
+--------+----------+--------+-------------+----------+
|    1001|2024-01-10|       1|            4|         1|
|    1002|2024-01-10|       3|            4|         4|
|    1007|2024-02-20|       1|            5|         5|
|    1015|2024-03-25|       1|            6|         6|
+--------+----------+--------+-------------+----------+


In [ ]:
# A moving average, and last() with the default frame vs the whole partition
w = W.partitionBy("customer_id").orderBy("order_date", "order_id")
orders.filter(F.col("customer_id") == 1).select(
    "order_id", "quantity",
    F.round(F.avg("quantity").over(w.rowsBetween(-1, W.currentRow)), 2).alias("avg_of_last_2"),
    F.last("order_id").over(w).alias("last_default_frame"),
    F.last("order_id").over(w.rowsBetween(W.unboundedPreceding, W.unboundedFollowing)).alias("last_of_partition"),
).orderBy("order_id").show()

+--------+--------+-------------+------------------+-----------------+
|order_id|quantity|avg_of_last_2|last_default_frame|last_of_partition|
+--------+--------+-------------+------------------+-----------------+
|    1001|       1|          1.0|              1001|             1015|
|    1002|       3|          2.0|              1002|             1015|
|    1007|       1|          2.0|              1007|             1015|
|    1015|       1|          1.0|              1015|             1015|
+--------+--------+-------------+------------------+-----------------+


### Exercises

**Q16 · `etl(orders)`** For every order of customers 1 and 3, show how many days passed since the same customer's previous order (`days_since_prev`, empty for the first order). Show customer_id, order_id, order_date and days_since_prev, sorted by customer_id, then order_date and order_id.

Expected output:

```
+-----------+--------+----------+---------------+
|customer_id|order_id|order_date|days_since_prev|
+-----------+--------+----------+---------------+
|          1|    1001|2024-01-10|           NULL|
|          1|    1002|2024-01-10|              0|
|          1|    1007|2024-02-20|             41|
|          1|    1015|2024-03-25|             34|
|          3|    1004|2024-02-03|           NULL|
|          3|    1009|2024-03-05|             31|
|          3|    1016|2024-03-28|             23|
+-----------+--------+----------+---------------+
```

In [ ]:
def etl(orders):
    # Write code here: return a DataFrame
    return orders

etl(orders).show()

**Q17 · `etl(orders)`** A running total: for customers 1 and 2, show customer_id, order_id, order_date, quantity and `items_so_far` (the items of this order and all earlier orders of the customer). Sort by customer_id, then order_date and order_id.

Expected output:

```
+-----------+--------+----------+--------+------------+
|customer_id|order_id|order_date|quantity|items_so_far|
+-----------+--------+----------+--------+------------+
|          1|    1001|2024-01-10|       1|           1|
|          1|    1002|2024-01-10|       3|           4|
|          1|    1007|2024-02-20|       1|           5|
|          1|    1015|2024-03-25|       1|           6|
|          2|    1003|2024-01-25|       2|           2|
|          2|    1005|2024-02-05|       1|           3|
|          2|    1013|2024-03-18|       1|           4|
+-----------+--------+----------+--------+------------+
```

In [ ]:
def etl(orders):
    # Write code here: return a DataFrame
    return orders

etl(orders).show()

**Q18 · `etl(orders, products)`** Show the delivered revenue (quantity × price) of each month and its growth against the previous month, in percent (1 decimal; empty for the first month). Columns: month, revenue, growth_pct; sorted by month.

Expected output:

```
+-----+-------+----------+
|month|revenue|growth_pct|
+-----+-------+----------+
|    1| 1545.0|      NULL|
|    2|  175.0|     -88.7|
|    3| 1395.0|     697.1|
+-----+-------+----------+
```

In [ ]:
def etl(orders, products):
    # Write code here: return a DataFrame
    return orders

etl(orders, products).show()

**Q19 · `etl(events)`** For customer 1's events (duplicates removed), show event_time, page and `prev_page` (the page of the previous event), and keep only the events where the page changed. The first event counts as a change. Sort by event_time.

Expected output:

```
+-------------------+-------+---------+
|         event_time|   page|prev_page|
+-------------------+-------+---------+
|2024-03-01 09:00:00|   home|     NULL|
|2024-03-01 09:10:00|product|     home|
|2024-03-01 11:00:00|   home|  product|
|2024-03-02 10:00:00|   cart|     home|
|2024-03-03 10:30:00|   home|     cart|
+-------------------+-------+---------+
```

In [ ]:
def etl(events):
    # Write code here: return a DataFrame
    return events

etl(events).show()

## Mixed problems
Like Zillacode's Medium and Hard problems: a join, an aggregation and a window together. Write down the steps first. About 15 minutes each.

**Q20 · `etl(orders, customers)`** In each city, find the customer with the most orders (on a tie, the smaller customer_id). Customers without a city are left out. Columns: city, name, orders; sorted by city.

Expected output:

```
+--------+-------------+------+
|    city|         name|orders|
+--------+-------------+------+
|  Berlin|Bella Schmidt|     3|
|  London|  Emma Wilson|     2|
|  Madrid|  Carlos Ruiz|     3|
|Tashkent| Aziz Karimov|     4|
+--------+-------------+------+
```

In [ ]:
def etl(orders, customers):
    # Write code here: return a DataFrame
    return orders

etl(orders, customers).show()

**Q21 · `etl(orders)`** For each customer with at least 2 orders, find the longest gap in days between two consecutive orders. Columns: customer_id, longest_gap_days; sorted by customer_id.

Expected output:

```
+-----------+----------------+
|customer_id|longest_gap_days|
+-----------+----------------+
|          1|              41|
|          2|              42|
|          3|              31|
|          4|              22|
|          5|              18|
+-----------+----------------+
```

In [ ]:
def etl(orders):
    # Write code here: return a DataFrame
    return orders

etl(orders).show()

**Q22 · `etl(orders, products)`** Rank the customers by delivered revenue (1 = the highest; the guest doesn't count) and show how much less each one spent than the customer ranked just above. Columns: customer_id, revenue, rank, gap_to_next_better; sorted by rank.

Expected output:

```
+-----------+-------+----+------------------+
|customer_id|revenue|rank|gap_to_next_better|
+-----------+-------+----+------------------+
|          2| 1525.0|   1|              NULL|
|          1| 1395.0|   2|             130.0|
|          5|   75.0|   3|            1320.0|
|          3|   60.0|   4|              15.0|
+-----------+-------+----+------------------+
```

In [ ]:
def etl(orders, products):
    # Write code here: return a DataFrame
    return orders

etl(orders, products).show()

**Q23 · `etl(orders, products)`** Which products were ordered but never delivered? Columns: product_id, product_name; sorted by product_id.

Expected output:

```
+----------+------------+
|product_id|product_name|
+----------+------------+
|       103|Coffee Maker|
+----------+------------+
```

In [ ]:
def etl(orders, products):
    # Write code here: return a DataFrame
    return orders

etl(orders, products).show()

## Answers
Try each exercise yourself first. Click an exercise to open its answer.

<details>
<summary><b>Q1</b> &nbsp;Which customers have never placed an order? Show customer_id and name.</summary>

```python
def etl(customers, orders):
    return customers.join(orders, "customer_id", "left_anti").select("customer_id", "name").orderBy("customer_id")

etl(customers, orders).show()
```

</details>

<details>
<summary><b>Q2</b> &nbsp;Show every customer with their number of orders, including customers with 0 orders. Show name and orders, ...</summary>

```python
def etl(customers, orders):
    return (customers.join(orders, "customer_id", "left")
                     .groupBy("customer_id", "name")
                     .agg(F.count("order_id").alias("orders"))
                     .select("name", "orders")
                     .orderBy(F.col("orders").desc(), "name"))

etl(customers, orders).show()
```

`count("order_id")` counts only matched rows, so Grace gets 0; `count("*")` would give her 1.

</details>

<details>
<summary><b>Q3</b> &nbsp;Show every product of the catalog and every product_id that appears in orders, with its number of orders (0 ...</summary>

```python
def etl(orders, products):
    counts = orders.groupBy("product_id").agg(F.count("*").alias("orders"))
    return (products.join(counts, "product_id", "full")
                    .select("product_id", "product_name", F.coalesce(F.col("orders"), F.lit(0)).alias("orders"))
                    .orderBy("product_id"))

etl(orders, products).show()
```

</details>

<details>
<summary><b>Q4</b> &nbsp;Which customers had all of their orders delivered (and at least one order)? Show their names, from A to Z.</summary>

```python
def etl(orders, customers):
    not_delivered = orders.filter(F.col("status") != "delivered").select("customer_id")
    return (customers.join(orders, "customer_id", "left_semi")
                     .join(not_delivered, "customer_id", "left_anti")
                     .select("name")
                     .orderBy("name"))

etl(orders, customers).show()
```

`left_semi` keeps customers with at least one order (once each), and `left_anti` then removes those with an order that isn't delivered.

</details>

<details>
<summary><b>Q5</b> &nbsp;Show every referral with both names: referral_date, referrer (the name of the customer who invited) and ...</summary>

```python
def etl(referrals, customers):
    referrer = customers.select(F.col("customer_id").alias("referrer_id"), F.col("name").alias("referrer"))
    referred = customers.select(F.col("customer_id").alias("referred_id"), F.col("name").alias("referred"))
    return (referrals.join(referrer, "referrer_id", "left")
                     .join(referred, "referred_id", "left")
                     .select("referral_date", "referrer", "referred")
                     .orderBy("referral_date"))

etl(referrals, customers).show()
```

</details>

<details>
<summary><b>Q6</b> &nbsp;Show the pairs of customers who live in the same city: city, customer_a and customer_b (names). Each pair ...</summary>

```python
def etl(customers):
    a, b = customers.alias("a"), customers.alias("b")
    return (a.join(b, (F.col("a.city") == F.col("b.city")) & (F.col("a.customer_id") < F.col("b.customer_id")))
             .select(F.col("a.city").alias("city"), F.col("a.name").alias("customer_a"), F.col("b.name").alias("customer_b"))
             .orderBy("city"))

etl(customers).show()
```

</details>

<details>
<summary><b>Q7</b> &nbsp;Compare customers with customers_new: which customers were added, removed or changed (a different name, email ...</summary>

```python
def etl(customers, customers_new):
    old, new = customers.alias("old"), customers_new.alias("new")
    same = (F.col("old.name").eqNullSafe(F.col("new.name"))
            & F.col("old.email").eqNullSafe(F.col("new.email"))
            & F.col("old.city").eqNullSafe(F.col("new.city")))
    return (old.join(new, F.col("old.customer_id") == F.col("new.customer_id"), "full")
               .select(F.coalesce(F.col("old.customer_id"), F.col("new.customer_id")).alias("customer_id"),
                       F.when(F.col("old.customer_id").isNull(), "added")
                        .when(F.col("new.customer_id").isNull(), "removed")
                        .when(~same, "changed").alias("change"))
               .filter(F.col("change").isNotNull())
               .orderBy("customer_id"))

etl(customers, customers_new).show()
```

`eqNullSafe` matters for customer 6: his city is null in both tables, and with `==` that comparison would be null instead of true.

</details>

<details>
<summary><b>Q8</b> &nbsp;For each delivered order, find the promotion of its product's category that was running on the order date. ...</summary>

```python
def etl(orders, products, promotions):
    return (orders.filter(F.col("status") == "delivered")
                  .join(products, "product_id")
                  .join(promotions, (products.category == promotions.category)
                                    & F.col("order_date").between(F.col("start_date"), F.col("end_date")))
                  .select("order_id", "product_name", "promo_id", "discount_pct")
                  .orderBy("order_id"))

etl(orders, products, promotions).show()
```

</details>

<details>
<summary><b>Q9</b> &nbsp;Make a table with every combination of customers 1 to 3 and every product category, with the number of orders ...</summary>

```python
def etl(customers, products, orders):
    combos = customers.filter(F.col("customer_id") <= 3).select("customer_id").crossJoin(products.select("category").distinct())
    counts = orders.join(products, "product_id").groupBy("customer_id", "category").agg(F.count("*").alias("orders"))
    return (combos.join(counts, ["customer_id", "category"], "left")
                  .select("customer_id", "category", F.coalesce(F.col("orders"), F.lit(0)).alias("orders"))
                  .orderBy("customer_id", "category"))

etl(customers, products, orders).show()
```

</details>

<details>
<summary><b>Q10</b> &nbsp;Show product_name, category, price and diff_from_avg (the price minus its category's average price, 2 ...</summary>

```python
def etl(products):
    w = W.partitionBy("category")
    return (products.select("product_name", "category", "price",
                            F.round(F.col("price") - F.avg("price").over(w), 2).alias("diff_from_avg"))
                    .orderBy("category", "price"))

etl(products).show()
```

</details>

<details>
<summary><b>Q11</b> &nbsp;For the orders of customers 1 and 2, show order_id, customer_id, quantity and pct_of_customer: the order's ...</summary>

```python
def etl(orders):
    w = W.partitionBy("customer_id")
    return (orders.filter(F.col("customer_id").isin(1, 2))
                  .select("order_id", "customer_id", "quantity",
                          F.round(F.col("quantity") / F.sum("quantity").over(w) * 100, 1).alias("pct_of_customer"))
                  .orderBy("order_id"))

etl(orders).show()
```

</details>

<details>
<summary><b>Q12</b> &nbsp;Show the most expensive product of each category: category, product_name and price, sorted by category.</summary>

```python
def etl(products):
    w = W.partitionBy("category").orderBy(F.col("price").desc())
    return (products.withColumn("rn", F.row_number().over(w))
                    .filter(F.col("rn") == 1)
                    .select("category", "product_name", "price")
                    .orderBy("category"))

etl(products).show()
```

</details>

<details>
<summary><b>Q13</b> &nbsp;Rank the customers who placed orders by their number of orders. Customers with the same number share a rank, ...</summary>

```python
def etl(orders, customers):
    counts = orders.groupBy("customer_id").agg(F.count("*").alias("orders"))
    w = W.orderBy(F.col("orders").desc())
    return (counts.join(customers, "customer_id")
                  .withColumn("rank", F.dense_rank().over(w))
                  .select("name", "orders", "rank")
                  .orderBy("rank", "name"))

etl(orders, customers).show()
```

</details>

<details>
<summary><b>Q14</b> &nbsp;Show the top 2 products of each category by items delivered (on a tie, the smaller product_id first). ...</summary>

```python
def etl(orders, products):
    items = (orders.filter(F.col("status") == "delivered")
                   .join(products, "product_id")
                   .groupBy("category", "product_id", "product_name")
                   .agg(F.sum("quantity").alias("items")))
    w = W.partitionBy("category").orderBy(F.col("items").desc(), "product_id")
    return (items.withColumn("rank", F.row_number().over(w))
                 .filter(F.col("rank") <= 2)
                 .select("category", "product_name", "items", "rank")
                 .orderBy("category", "rank"))

etl(orders, products).show()
```

</details>

<details>
<summary><b>Q15</b> &nbsp;Show each customer's latest order: customer_id, order_id, order_date and status (on the same day, the larger ...</summary>

```python
def etl(orders):
    w = W.partitionBy("customer_id").orderBy(F.col("order_date").desc(), F.col("order_id").desc())
    return (orders.filter(F.col("customer_id").isNotNull())
                  .withColumn("rn", F.row_number().over(w))
                  .filter(F.col("rn") == 1)
                  .select("customer_id", "order_id", "order_date", "status")
                  .orderBy("customer_id"))

etl(orders).show()
```

</details>

<details>
<summary><b>Q16</b> &nbsp;For every order of customers 1 and 3, show how many days passed since the same customer's previous order ...</summary>

```python
def etl(orders):
    w = W.partitionBy("customer_id").orderBy("order_date", "order_id")
    return (orders.filter(F.col("customer_id").isin(1, 3))
                  .withColumn("days_since_prev", F.datediff("order_date", F.lag("order_date").over(w)))
                  .select("customer_id", "order_id", "order_date", "days_since_prev")
                  .orderBy("customer_id", "order_date", "order_id"))

etl(orders).show()
```

</details>

<details>
<summary><b>Q17</b> &nbsp;A running total: for customers 1 and 2, show customer_id, order_id, order_date, quantity and items_so_far ...</summary>

```python
def etl(orders):
    w = (W.partitionBy("customer_id").orderBy("order_date", "order_id")
          .rowsBetween(W.unboundedPreceding, W.currentRow))
    return (orders.filter(F.col("customer_id").isin(1, 2))
                  .withColumn("items_so_far", F.sum("quantity").over(w))
                  .select("customer_id", "order_id", "order_date", "quantity", "items_so_far")
                  .orderBy("customer_id", "order_date", "order_id"))

etl(orders).show()
```

</details>

<details>
<summary><b>Q18</b> &nbsp;Show the delivered revenue (quantity × price) of each month and its growth against the previous month, in ...</summary>

```python
def etl(orders, products):
    monthly = (orders.filter(F.col("status") == "delivered")
                     .join(products, "product_id")
                     .groupBy(F.month("order_date").alias("month"))
                     .agg(F.sum(F.col("quantity") * F.col("price")).alias("revenue")))
    previous = F.lag("revenue").over(W.orderBy("month"))
    return (monthly.withColumn("growth_pct", F.round((F.col("revenue") - previous) / previous * 100, 1))
                   .orderBy("month"))

etl(orders, products).show()
```

</details>

<details>
<summary><b>Q19</b> &nbsp;For customer 1's events (duplicates removed), show event_time, page and prev_page (the page of the previous ...</summary>

```python
def etl(events):
    w = W.orderBy("event_time")
    return (events.filter(F.col("customer_id") == 1)
                  .distinct()
                  .withColumn("prev_page", F.lag("page").over(w))
                  .filter(F.col("prev_page").isNull() | (F.col("page") != F.col("prev_page")))
                  .select("event_time", "page", "prev_page")
                  .orderBy("event_time"))

etl(events).show()
```

`page != prev_page` is null for the first event, so it needs its own `isNull()` test.

</details>

<details>
<summary><b>Q20</b> &nbsp;In each city, find the customer with the most orders (on a tie, the smaller customer_id). Customers without a ...</summary>

```python
def etl(orders, customers):
    counts = orders.groupBy("customer_id").agg(F.count("*").alias("orders"))
    w = W.partitionBy("city").orderBy(F.desc("orders"), "customer_id")
    return (customers.filter(F.col("city").isNotNull())
                     .join(counts, "customer_id")
                     .withColumn("rn", F.row_number().over(w))
                     .filter(F.col("rn") == 1)
                     .select("city", "name", "orders")
                     .orderBy("city"))

etl(orders, customers).show()
```

</details>

<details>
<summary><b>Q21</b> &nbsp;For each customer with at least 2 orders, find the longest gap in days between two consecutive orders. ...</summary>

```python
def etl(orders):
    w = W.partitionBy("customer_id").orderBy("order_date", "order_id")
    return (orders.filter(F.col("customer_id").isNotNull())
                  .withColumn("gap_days", F.datediff("order_date", F.lag("order_date").over(w)))
                  .groupBy("customer_id")
                  .agg(F.max("gap_days").alias("longest_gap_days"))
                  .filter(F.col("longest_gap_days").isNotNull())
                  .orderBy("customer_id"))

etl(orders).show()
```

A customer with one order has only a null gap, and `max` of nulls is null: that's how the filter removes them.

</details>

<details>
<summary><b>Q22</b> &nbsp;Rank the customers by delivered revenue (1 = the highest; the guest doesn't count) and show how much less ...</summary>

```python
def etl(orders, products):
    revenue = (orders.filter((F.col("status") == "delivered") & F.col("customer_id").isNotNull())
                     .join(products, "product_id")
                     .groupBy("customer_id")
                     .agg(F.sum(F.col("quantity") * F.col("price")).alias("revenue")))
    w = W.orderBy(F.desc("revenue"))
    return (revenue.withColumn("rank", F.rank().over(w))
                   .withColumn("gap_to_next_better", F.lag("revenue").over(w) - F.col("revenue"))
                   .orderBy("rank"))

etl(orders, products).show()
```

</details>

<details>
<summary><b>Q23</b> &nbsp;Which products were ordered but never delivered? Columns: product_id, product_name; sorted by product_id.</summary>

```python
def etl(orders, products):
    ordered = orders.select("product_id").distinct()
    delivered = orders.filter(F.col("status") == "delivered").select("product_id")
    return (ordered.join(delivered, "product_id", "left_anti")
                   .join(products, "product_id")
                   .select("product_id", "product_name")
                   .orderBy("product_id"))

etl(orders, products).show()
```

</details>